# Watershed Study Area Map Making and Geospatial Summary Statistic Generation

This script both plots and calculates basic metrics of physical and climate watershed properties across the study area

**Note: This is mainly code from Sage that has been condensed and edited.**

In [8]:
# Colab check
import sys
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
  from google.colab import drive
  drive.mount('/content/drive')
  %cd /content/drive/MyDrive/Python Code/Paper-condensed/scripts/
else:
  %cd /Users/sagefox/My Drive (sagefox@uw.edu)/Research/Python_Code/Paper-condensed/scripts/

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/.shortcut-targets-by-id/1jKcNrw1UEQ_p8vbILfPLdYnIma0uaqCy/Python_Code/Paper-condensed/scripts


In [ ]:
savefig = True
data_path = Path.cwd().parent/"data"/"geospatial"
fig_path = Path.cwd().parent/"results"

## Import Required Packages

In [9]:
######## IMPORTING ########

#Imports first, if necessary
#pip install geopandas rioxarray xarray rasterio cartopy matplotlib pooch
import os
import requests
from pathlib import Path
import warnings
import colorsys

import numpy as np
import pandas as pd
import xarray as xr

import geopandas as gpd
import rasterio as rio
import rasterstats
import rioxarray as rxr


from rasterio import plot, mask
from rasterio.plot import show
from rasterio.warp import calculate_default_transform, reproject, Resampling

import pyproj
from pyproj import CRS, Transformer

import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.mpl.ticker import LatitudeFormatter, LongitudeFormatter

from shapely.geometry import mapping
from shapely.geometry import box
from shapely.geometry import Point

import matplotlib.pyplot as plt
import matplotlib.cm as cm
from matplotlib.cm import ScalarMappable
import matplotlib.ticker as mticker
import matplotlib.patches as mpatches
import matplotlib.lines as mlines
from matplotlib_scalebar.scalebar import ScaleBar
import matplotlib.colors as mcolors
from matplotlib.colors import to_rgb, to_hex
from matplotlib.colors import LightSource

from pysheds.grid import Grid

import pooch

warnings.filterwarnings('ignore')

In [10]:
###### FOR CITATIONS ONLY, THIS PRINTS VERSIONS USED ######

import importlib

# Define a list of packages with version attributes
packages = {
    "matplotlib": "matplotlib",
    "numpy": "numpy",
    "pandas": "pandas",
    "geopandas": "geopandas",
    "rasterio": "rasterio",
    "cartopy": "cartopy",
    "pyproj": "pyproj",
    "shapely": "shapely",
    "rasterstats": "rasterstats",
    "xarray": "xarray",
    "rioxarray": "rioxarray",
    "matplotlib_scalebar": "matplotlib_scalebar",
    "pysheds": "pysheds",
    "pooch": "pooch"
}

print("Package Versions:\n" + "-"*40)
for name, module_name in packages.items():
    try:
        module = importlib.import_module(module_name)
        version = getattr(module, '__version__', 'Version info not available')
        print(f"{name:20}: {version}")
    except ImportError:
        print(f"{name:20}: Not installed")

Package Versions:
----------------------------------------
matplotlib          : 3.10.0
numpy               : 2.0.2
pandas              : 2.2.2
geopandas           : 1.1.1
rasterio            : 1.4.3
cartopy             : 0.25.0
pyproj              : 3.7.2
shapely             : 2.1.1
rasterstats         : 0.20.0
xarray              : 2025.9.0
rioxarray           : 0.19.0
matplotlib_scalebar : 0.9.0
pysheds             : 0.5
pooch               : v1.8.2


## Prepare Files

In [11]:
####### PROJECTION PREP #############

# Set CRS proj string to an equal area projection centered on the study area (looked up on Projection Wizard: https://projectionwizard.org/)
#crs_proj_string = "+proj=tcea +lon_0=-72.7294922 +datum=WGS84 +units=m +no_defs"

crs_proj_string = "+proj=aea +lon_0=-73.8061523 +lat_1=-46.0556595 +lat_2=-42.128073 +lat_0=-44.0918663 +datum=WGS84 +units=m +no_defs"
crs_proj_string

'+proj=aea +lon_0=-73.8061523 +lat_1=-46.0556595 +lat_2=-42.128073 +lat_0=-44.0918663 +datum=WGS84 +units=m +no_defs'

In [12]:
####### GAUGE STATION & NOTABLE POINTS COORDINATE PREP ########

### Set path for gauge station coordinates
file_path = data_path/"gauge_station_key.xlsx"
file_path_2 = data_path/"notable_points.xlsx"

# Read the gauge station xlsx file into a pandas DataFrame
gs_df = pd.read_excel(file_path)
# Read in other notable points file
gs_df_2 = pd.read_excel(file_path_2)

# Convert gauge station df into a GeoDataFrame
gdf_points = gpd.GeoDataFrame(
    gs_df,
    geometry=gpd.points_from_xy(gs_df['gauge_lon'], gs_df['gauge_lat']),
    crs="EPSG:4326"  # or whichever lat/lon CRS it is
)
# Do the same thing for other notable points
gdf_points_2 = gpd.GeoDataFrame(
    gs_df_2,
    geometry=gpd.points_from_xy(gs_df_2['lon'], gs_df_2['lat']),
    crs="EPSG:4326"  # or whichever lat/lon CRS it is
)

# Reproject to match the watershed boundary
gdf_points = gdf_points.to_crs(crs_proj_string)
gdf_points_2 = gdf_points_2.to_crs(crs_proj_string)

In [13]:
# ####### WATER SAMPLING POINTS PREP PART 1 ###############

# # set paths
# file = "../data/geospatial/FSEQpts.shp"

# # make geojson file
# shapefile = gpd.read_file(file)
# filename = os.path.basename(file).split('/')[-1].split('.')[0]
# shapefile.to_file(f"../../../data/geospatial/raw/geojson_files/{filename}.geojson")

# # unnecessary

In [14]:
####### WATER SAMPLING POINTS PREP PART 2 ###############

# load new sampling points geojson file
sampling_gdf = gpd.read_file(data_path/"FSEQpts.geojson")

# Reproject to appropriate CRS project string
sampling_gdf = sampling_gdf.to_crs(crs_proj_string)

#sampling_gdf.crs

In [15]:
######## WATERSHED OUTLET POINTS PREP - IF NEEDED #####
outlets_gdf = gpd.read_file(data_path/"Basins_Patagonia_mouth.geojson")
outlets_gdf = outlets_gdf.to_crs(crs_proj_string)

In [16]:
# ####### OUTLET FLOW PREP PART 1 ###############

# # set path
# flow_path = "../data/geospatial/raw/shapefiles/Watershed_Outlet_Flow/Basins_Patagonia_mouth.shp"

# # make geojson file
# shapefile = gpd.read_file(flow_path)
# filename = os.path.basename(flow_path).split('/')[-1].split('.')[0]
# shapefile.to_file(f"../../../data/geospatial/raw/geojson_files/{filename}.geojson")

# # Unnecessary

In [17]:
# ####### OUTLET FLOW PREP PART 2 ###############

# #NOTE: This is not being used yet, because I would need to index into the correct outlets, this plots all 500

# # load new sampling points geojson file
# flow_gdf = gpd.read_file("../data/geospatial/Basins_Patagonia_mouth.geojson")

# # Reproject to appropriate CRS project string
# flow_gdf = flow_gdf.to_crs(crs_proj_string)

In [21]:
########### WATERSHED BOUNDARY PREP ##########

# Load watershed boundary geometry - make sure to use the projected geometry
ws_gdf = gpd.read_file(data_path/"all_watersheds_projected.geojson")

# Reproject to appropriate CRS project string
ws_gdf_proj = ws_gdf.to_crs(crs_proj_string)

# Capitalize river names just for formatting style in plotting
ws_gdf_proj["geometry_name"] = ws_gdf_proj["geometry_name"].str.capitalize()

# Add new columns to the dataframe for later use in plotting, which set coordinates for name display within each watershed
ws_gdf_proj["label_x"] = ws_gdf_proj.geometry.centroid.x
ws_gdf_proj["label_y"] = ws_gdf_proj.geometry.centroid.y

# Change order of entries to be North-South, so that they match excluded watershed boudary dataframe
#for use in computing excluded area
desired_order = ["Puelo", "Yelcho", "Palena", "Cisnes", "Aysen"] #This is north to south, not alphabetical

ws_gdf_proj = (
    ws_gdf_proj
    .set_index("geometry_name")        # 1) set geometry_name as index
    .loc[desired_order]                # 2) re-index in the order we want
    .reset_index(drop=False)           # 3) restore geometry_name as a column
)

# Compute total area of all watersheds, dividing by 1E6 to convert sq. m to sq. km,
# Assigning output to "total_area" object
total_area = ws_gdf_proj.area/1E6

# Check this new total_area dataframe if needed
#total_area

### Check watershed boundary gdf if needed
#ws_gdf_proj

## Check CRS on watershed boundaries if needed
#ws_gdf_proj.crs

In [22]:
# ########### GEOLOGY AND SOILS LAYER PREP 1 #####

# # set shapefile paths
# geology_shp_key = '../../../data/geospatial/raw/shapefiles/geology/geo_leyenda.shp'
# geology_shp = '../../../data/geospatial/raw/shapefiles/geology/geologiaWSPAT.shp'
# geo_arg_zone1 = '../../../data/geospatial/raw/shapefiles/geology/geo_arg_zone1.shp'
# geo_arg_zone3 = '../../../data/geospatial/raw/shapefiles/geology/geo_arg_zone3.shp'
# soils_shp = '../../../data/geospatial/raw/shapefiles/soils/suelosleyenda.shp'

# # make geojson files from shapefiles

# #geology
# shapefile_geo = gpd.read_file(geology_shp)
# filename = os.path.basename(geology_shp).split('/')[-1].split('.')[0]
# shapefile_geo.to_file("../../../data/geospatial/raw/geojson_files/geology.geojson")

# #geology
# shapefile_geo = gpd.read_file(geo_arg_zone1)
# filename = os.path.basename(geo_arg_zone1).split('/')[-1].split('.')[0]
# shapefile_geo.to_file("../../../data/geospatial/raw/geojson_files/geo_arg_zone1.geojson")

# #geology
# shapefile_geo = gpd.read_file(geo_arg_zone3)
# filename = os.path.basename(geo_arg_zone3).split('/')[-1].split('.')[0]
# shapefile_geo.to_file("../../../data/geospatial/raw/geojson_files/geo_arg_zone3.geojson")

# # #geology
# # shapefile_geo = gpd.read_file(geology_shp_key)
# # filename = os.path.basename(geology_shp_key).split('/')[-1].split('.')[0]
# # shapefile_geo.to_file("../../../data/geospatial/raw/geojson_files/geology_key.geojson")

# #soils
# shapefile = gpd.read_file(soils_shp)
# filename = os.path.basename(file).split('/')[-1].split('.')[0]
# shapefile.to_file("../../../data/geospatial/raw/geojson_files/soils.geojson")

# # unnecessary

In [23]:
########### GEOLOGY AND SOILS LAYER PREP 2 ##### ***IN PROGRESS, UNFINISHED

# Load new geojson files and reproject to your custom CRS
geology_gdf = gpd.read_file(data_path/"geology.geojson")
geology_gdf = geology_gdf.to_crs(crs_proj_string)

geology_key_gdf = gpd.read_file(data_path/"geology_key.geojson")
geology__key_gdf = geology_key_gdf.to_crs(crs_proj_string)

geo_arg_zone1_gdf = gpd.read_file(data_path/"geo_arg_zone1.geojson")
geo_arg_zone1_gdf = geo_arg_zone1_gdf.to_crs(crs_proj_string)

geo_arg_zone3_gdf = gpd.read_file(data_path/"geo_arg_zone3.geojson")
geo_arg_zone3_gdf = geo_arg_zone3_gdf.to_crs(crs_proj_string)

soils_gdf = gpd.read_file(data_path/"soils.geojson")
soils_gdf = soils_gdf.to_crs(crs_proj_string)

# Trim (clip) to study area using ws_gdf_proj.
# This assumes ws_gdf_proj is a GeoDataFrame with your watershed boundaries.
geology_gdf_clipped = geology_gdf.clip(ws_gdf_proj)
geology_key_gdf_clipped = geology_key_gdf.clip(ws_gdf_proj)
geo_arg_zone1_clipped = geo_arg_zone1_gdf.clip(ws_gdf_proj)
geo_arg_zone3_clipped = geo_arg_zone3_gdf.clip(ws_gdf_proj)
soils_gdf_clipped = soils_gdf.clip(ws_gdf_proj)

# (Optional) Check the results
print("Geology clipped shape:", geology_gdf_clipped.shape)
print("Geology key clipped shape:", geology_key_gdf_clipped.shape)
print("Soils clipped shape:", soils_gdf_clipped.shape)

Geology clipped shape: (833, 10)
Geology key clipped shape: (0, 6)
Soils clipped shape: (264, 14)


In [36]:
####### MERGE THE THREE SEPARATE GEOLOGY GDFS, COMBINING GEOLOGY CLASSIFICATION COLUMNS, THEN PLOT ########

# Concatenate the GeoDataFrames
geo_gdf = pd.concat([geology_gdf_clipped, geo_arg_zone1_clipped, geo_arg_zone3_clipped], ignore_index=True)

# Convert to a GeoDataFrame (if not already)
geo_gdf = gpd.GeoDataFrame(geo_gdf, geometry='geometry')

# Optional: ensure the CRS is set (they all have the same CRS)
geo_gdf.set_crs(geology_gdf_clipped.crs, inplace=True)

# Assuming merged_gdf is your merged GeoDataFrame
# Create a new column 'geology_class' that takes values from 'UNIDAD_GEO'
# if available, otherwise from 'litologia'
geo_gdf['geology_class'] = geo_gdf['UNIDAD_GEO'].combine_first(geo_gdf['litologia'])

# Alternatively, you could do:
# merged_gdf['geology_class'] = merged_gdf['UNIDAD_GEO'].fillna(merged_gdf['litologia'])

# Plot the merged GeoDataFrame with colors based on the geology_class values
# fig, ax = plt.subplots(figsize=(10, 10))
# geo_gdf.plot(
#     column='geology_class',
#     categorical=True,
#     legend=False,
#     ax=ax,
#     cmap='Set3'  # you can change the colormap as desired
# )

# ax.set_title("Merged Geology Classification")
# ax.set_xlabel("Longitude")
# ax.set_ylabel("Latitude")
# plt.show()

In [27]:
# ###### THIS CELL USES THE GEOLOGY KEY FOR THE CHILEAN DATABASE TO SWAP CODES FOR CLASSES IN THE GEOLOGY CLASS COLUMN ######

# # Load the key Excel file that contains the mapping. The variable `key` contains its path.
# key_df = pd.read_excel("../data/geospatial/code_GEOunique.xlsx")

# # Create a dictionary mapping from CodeGEO to clasificacion.
# code_to_description = pd.Series(key_df['clasificacion'].values, index=key_df['CodeGEO']).to_dict()

# # Replace entries in the 'geology_class' column: codes that are found in the mapping will be swapped for their verbal descriptions.
# geo_gdf['geology_class'] = geo_gdf['geology_class'].replace(code_to_description)

# geo_gdf[['geology_class']].to_excel("../data/geospatial/common_code.xlsx", index=False)

In [51]:
key_ser = pd.read_excel(data_path/"code_GEOunique.xlsx", usecols=["CodeGEO", "clasificacion"], index_col=0)['clasificacion']
geo_gdf[['geology_class']].replace(key_ser, inplace=True)

In [35]:
#geo_gdf['geology_class'] = pd.read_excel("../data/geospatial/common_code.xlsx")['geology_class']

In [52]:
######## REASSIGN GEOLOGY CLASSES TO SIMPLER CLASSES BASED ON KEYWORDS (CAN ADJUST) FOR PLOTTING
###### THIS IS NECESSARY TO MERGE THE TWO GEOLOGY DATABASES

def assign_english_geology_category(desc):
    """
    Standardizes a Spanish geology description into one of several English categories:
       - "Intrusive"
       - "Metamorphic"
       - "Volcanic-Sedimentary"
       - "Volcanic"
       - "Sedimentary"
       - "Other"
    """
    d = str(desc).strip().lower()

    # Keyword lists
    intrusive_keywords = ["granito", "diorito", "gabro", "porfid", "intrus"]
    metamorphic_keywords = ["metam", "esquiz", "metapel", "metachert", "metabas", "neis", "ultra", "gneas"]  # Including 'gneas' as a clue when combined with metam
    sedimentary_keywords = ["sedim", "arenisc", "lutit", "conglomer", "caliz", "evapor", "coquina", "silic"]

    # Intrusive: if any intrusive keyword is found.
    if any(kw in d for kw in intrusive_keywords):
        return "Intrusive"

    # Metamorphic: if any metamorphic keyword is found.
    # (This will catch entries with "metam", even if other igneous descriptors are present.)
    if any(kw in d for kw in metamorphic_keywords):
        return "Metamorphic"

    # Mixed volcanic-sedimentary:
    if ("volc" in d or "piro" in d) and any(kw in d for kw in sedimentary_keywords):
        return "Volcanic-Sedimentary"

    # Pure volcanic (or igneous) signals:
    if "volc" in d or "piro" in d or "gneas" in d:
        return "Volcanic"

    # Sedimentary:
    if any(kw in d for kw in sedimentary_keywords):
        return "Sedimentary"

    return "Other"

# Apply the grouping function to the "geology_class" column.
geo_gdf["geology_group_english"] = geo_gdf["geology_class"].apply(assign_english_geology_category)

# (Optional) Display a summary showing unique original descriptions and the assigned groups.
unique_mapping = geo_gdf[["geology_class", "geology_group_english"]].drop_duplicates().sort_values(by="geology_class")
#print("Mapping of original geology_class to standardized groups:")
#print(unique_mapping)

In [53]:
######## MORE GEOLOGY PREP #####

# Define the desired order.
geology_order = ["Intrusive", "Metamorphic", "Volcanic-Sedimentary", "Volcanic", "Sedimentary", "Other"]

# Convert the column to a categorical with this specific order.
geo_gdf["geology_group_english"] = pd.Categorical(geo_gdf["geology_group_english"], categories=geology_order, ordered=True)

# Create a numeric code column from the categorical data.
geo_gdf["geology_code"] = geo_gdf["geology_group_english"].cat.codes

cmap_geol = ["#8ed3c7", "#ffffb4", "#bebada", "#fb8072", "#7fb1d3", "#fdb462"]

In [54]:
############ DEM PREP #################

### Read in the DEM with rxr
dem_da = rxr.open_rasterio(data_path/"trimmed_dem_projected.tif",mask_and_scale=True).squeeze()

# reproject the DEM to the crs proj string specified above
dem_da = dem_da.rio.reproject(crs_proj_string)

## Clip the projected dem by watershed boundaries, making sure "mapping" is installed
dem_da_clipped = dem_da.rio.clip(
    ws_gdf_proj.geometry.apply(mapping),  # polygons to clip by
    ws_gdf_proj.crs,                     # CRS of those polygons
    drop=True                            # drop outside values
)


In [55]:
########## HILLSHADE PREP ###################

# 1) Convert dem_da_clipped to a numpy array
dem_arr = dem_da.values

# 2) Extract resolution (in projected units, e.g. meters)
xres, yres = dem_da.rio.resolution()
xres, yres = abs(xres), abs(yres)  # ensure positive spacing

# 3) Compute gradients in y and x directions
gy, gx = np.gradient(dem_arr, yres, xres)

# 4) Calculate slope (radians) from the horizontal
#    slope = π/2 - arctan( sqrt(gx^2 + gy^2) )
slope = np.pi/2.0 - np.arctan(np.sqrt(gx**2 + gy**2))

# 5) Calculate aspect (radians), measured clockwise from North
#    Often aspect = arctan2( -gx, gy ), but you can adjust sign if needed
aspect = np.arctan2(-gx, gy)

# 6) Define sun angles (azimuth & altitude in degrees)
azimuth_deg = 315.0   # sun coming from NW
altitude_deg = 45.0   # angle above horizon
azimuth_rad = np.deg2rad(azimuth_deg)
altitude_rad = np.deg2rad(altitude_deg)

# 7) Apply Lambertian hillshade formula
hs_arr = (np.sin(altitude_rad) * np.sin(slope)
          + np.cos(altitude_rad) * np.cos(slope) * np.cos(azimuth_rad - aspect))

# Clip negative (no illumination) to zero => dark shadow
hs_arr = np.where(hs_arr < 0, 0, hs_arr)

# 8) Convert the hillshade numpy array back to a DataArray with the same coords/CRS as dem_da_clipped
hillshade_da_reproj = dem_da.copy(data=hs_arr)
hillshade_da_reproj.name = "hillshade"

#print("Hillshade created! Shape:", hillshade_da_reproj.shape)
#print("Hillshade CRS:", hillshade_da_reproj.rio.crs)

In [56]:
########### SLOPE MAP PREP #####################################

# 1) Extract DEM values as a NumPy array if not already done in previous cell
dem_arr = dem_da_clipped.values

# 2) Get the resolution (assumes a projected CRS in meters)
xres, yres = dem_da_clipped.rio.resolution()
xres, yres = abs(xres), abs(yres)  # ensure positive spacing

# 3) Compute gradients in the y & x directions
gy, gx = np.gradient(dem_arr, yres, xres)

# 4) Calculate slope in radians (from the horizontal)
#    slope = arctan( sqrt(gx^2 + gy^2) )
slope_radians = np.arctan(np.sqrt(gx**2 + gy**2))

# 5) Convert slope to degrees
slope_degrees = np.degrees(slope_radians)

# 6) Make a new DataArray with slope values
slope_da = dem_da_clipped.copy(data=slope_degrees)
slope_da.name = "slope_degrees"

# Diagnostic printing if needed
#print("Slope map created. Range:", slope_da.min().values, "to", slope_da.max().values, "degrees")

In [57]:
########### LAND COVER LAYER PREP (COLLAPSED CLASSES) #########################

# Path to your landcover file
lc_path = data_path/"landCover_trimmed2.tif"

# 1) Open land cover as a rioxarray DataArray, masking invalid pixels.
lc_da = rxr.open_rasterio(lc_path, masked=True)
# If it's single-band, remove the "band" dimension.
if "band" in lc_da.dims:
    lc_da = lc_da.squeeze("band", drop=True)

# 2) Reproject land cover to match your TCEA-based CRS.
lc_da_reproj = lc_da.rio.reproject(crs_proj_string)

# 3) Clip to watershed boundaries.
lc_da_clipped = lc_da_reproj.rio.clip(
    ws_gdf_proj.geometry.apply(mapping),
    ws_gdf_proj.crs,
    drop=True
)

# 4) Collapse land cover classes.
# Original dictionary (for reference):
# {20: 'Shrub', 30: 'Herb. veg', 40: 'Agriculture', 50: 'Urban',
#  60: 'Bare/sparse veg', 70: 'Snow & Ice', 80: 'Open water', 90: 'Wetland',
#  112: 'Everg forest (c)', 114: 'Decid forest (c)', 115: 'Mixed forest (c)',
#  116: 'Unk forest (c)', 122: 'Everg forest (o)', 124: 'Decid forest (o)',
#  126: 'Unk forest (o)'}
#
# Desired grouping:
# - Merge all forest categories into one ("Forest"): 112, 114, 115, 116, 122, 124, 126
# - Merge "Shrub" and "Herb. veg" into one ("Pampas"): 20, 30
# - Merge "Agriculture" and "Urban" into one ("Developed"): 40, 50
# - Merge "Bare/sparse veg" and "Snow & Ice" into one ("Snow/Barren"): 60, 70
# - Merge "Open water" and "Wetland" into one ("Water"): 80, 90

# Create a new DataArray for the collapsed land cover.
# (We use the original values from lc_da_clipped for the classification.)
lc_da_collapsed = lc_da_clipped.copy()

lc_da_collapsed = xr.where(lc_da_clipped.isin([112, 114, 115, 116, 122, 124, 126]), 1, lc_da_collapsed)
lc_da_collapsed = xr.where(lc_da_clipped.isin([20, 30]), 2, lc_da_collapsed)
lc_da_collapsed = xr.where(lc_da_clipped.isin([40, 50]), 3, lc_da_collapsed)
lc_da_collapsed = xr.where(lc_da_clipped.isin([60, 70]), 4, lc_da_collapsed)
lc_da_collapsed = xr.where(lc_da_clipped.isin([80, 90]), 5, lc_da_collapsed)
#lc_da_collapsed = xr.where(lc_da_clipped.isin([50]), 6, lc_da_collapsed)

# 5) Create an associated dictionary for the collapsed classes.
landcover_key = {
    1: "       Forest",
    2: "   Cold Steppe",
    3: "  Agriculture",
    4: "Bare/Snow/Ice",
    5: "        Water"
#    6: "     Urban"
}

########### DEFINE COLORMAP FOR COLLAPSED LAND COVER ###########
# Here are example colors for your five collapsed classes:
# 1: Forest, 2: Shrublands, 3: Agriculture, 4: Snow/Ice, 5: Water, 6: Urban.

#Note, this next line is the original color scheme set for just five categories, where ag and urban were combined into "developed"
#colors = ['#006400', '#FFD700', '#FF8C00', '#D3D3D3', '#0000FF']
#colors = ['#009E73', '#F0E442', '#E69F00', '#D3D3D3', '#0072B2', '#D55E00']
colors = ['#006400', '#FFD700', '#FF8C00', '#D3D3D3', '#0000FF']


cmap_collapsed = mcolors.ListedColormap(colors)
# Set boundaries so that values 1,2,...,5 are centered within intervals.
norm = mcolors.BoundaryNorm(boundaries=[0.5, 1.5, 2.5, 3.5, 4.5, 5.5], ncolors=len(colors))

# (Optional) Print unique values and the key to verify.
#print("Unique collapsed classes:", np.unique(lc_da_collapsed.values))
print("Landcover key:", landcover_key)

Landcover key: {1: '       Forest', 2: '   Cold Steppe', 3: '  Agriculture', 4: 'Bare/Snow/Ice', 5: '        Water'}


In [60]:
########### CLIMATE LAYER PREP #########################

#open files using rioxarray
precr = rxr.open_rasterio(data_path/"prec_coarse.nc",
                       chunks = 'auto', masked = True).squeeze()

#Reproject and clip
prec = precr.rio.write_crs('epsg:4326')                          #write an arbitrary crs - maybe look into this later
prec_repr = prec.rio.reproject(crs_proj_string, resampling=3)       #reproject
prec_clip = prec_repr.rio.clip(ws_gdf_proj.geometry)               #clip to watershed area
prec_clip_annual = (prec_clip.mean(dim = 'time')*365) # make the annual total precip values

In [61]:
################ EXLUDED CATCHMENTS BOUNDARY PREP ################

# #load shapefile with both full and exclusion watersheds
# file = "../data/geospatial/Basins_sampling_points_data.shp"

# #convert shapefile to geojson file, saving that to a new folder
# shapefile = gpd.read_file(file)
# filename = os.path.basename(file).split('/')[-1].split('.')[0]
# shapefile.to_file(f"../../../data/geospatial/raw/geojson_files/{filename}.geojson")
# # unnecessary

# Load watershed boundary geometry as geojson
ws_exclude_gdf = gpd.read_file(f"../data/geospatial/Basins_sampling_points_data.geojson")

# Reproject to appropriate CRS project string
ws_exclude_gdf = ws_exclude_gdf.to_crs(crs_proj_string)

# Filter rows where gauge_id ends with "WQ", selecting only the excluded catchments
ws_exclude_gdf_wq_only = ws_exclude_gdf[ws_exclude_gdf["gauge_id"].str.endswith("WQ")]

### calculate areas of each excluded catchment, dividing by 1E6 to convert sq m to sq km.
#Save to excluded_area object for later use
excluded_area = ws_exclude_gdf_wq_only.area/1E6

# reset the index to match corresponding full watershed dataframe
excluded_area = excluded_area.reset_index(drop=True)

##double check new CRS on watershed boundaries if needed
#ws_exclude_gdf.crs

#check index and areas of excluded catchments to ensure it worked
#excluded_area

In [62]:
# ####### FUTA DAM EXCLUSION ZONE PREP ##########

# # Load watershed boundary geometry as geojson
# futa_dam_exclude = gpd.read_file(f"../../../data/geospatial/raw/geojson_files/watershed_boundaries/futa_dam.json")

# # Reproject to appropriate CRS project string
# futa_dam_exclude = futa_dam_exclude.to_crs(crs_proj_string)

# # Currently unnecessary

In [63]:
# ################ OTHER SHAPEFILE PREP: LAGO YELCHO ################

# # Load watershed boundary geometry as geojson
# lago_yelcho_gdf = gpd.read_file(f"../../../data/geospatial/raw/geojson_files/yelcho_lake.geojson")

# # Reproject to appropriate CRS project string
# lago_yelcho_gdf = lago_yelcho_gdf.to_crs(crs_proj_string)

# ### calculate areas of each excluded catchment, dividing by 1E6 to convert sq m to sq km.
# #Save to excluded_area object for later use
# lago_yelcho_area = lago_yelcho_gdf.area/1E6

# ##double check new CRS on watershed boundaries if needed
# #lago_yelcho_gdf.crs

# #check index and areas of excluded catchments to ensure it worked
# print(f'Lago Yelcho Area: {lago_yelcho_area[0]} sq. km.')

## Compute Geospatial Statistics

In [64]:
########### CALCULATE CATCHMENT AREA  STATS AND SAVE AS CSV ########

# Prepare a list to store results
results_list = []

# Prepare a list of river names from North to South (to match formatting of dataframes above)
names_list = ["Puelo", "Yelcho", "Palena", "Cisnes", "Aysen"]

for i in range(5):
    # Grab the current watershed name from the names_list object
    river_name = names_list[i]

    # Grab the already prepared area values (in square meters or appropriate unit)
    total_area_current = total_area[i]
    excluded_area_current = excluded_area[i]

    # Compute the percentage of the watershed excluded
    pct_excluded = (excluded_area_current / total_area_current) * 100

    # Save results in a dictionary with the desired column headers
    results_list.append({
        "river_name": river_name,
        "total_area": total_area_current,
        "excluded_area": excluded_area_current,
        "pct_excluded": pct_excluded
    })

# Convert the list of dictionaries into a DataFrame
results_df = pd.DataFrame(results_list)

# # Save the DataFrame as a CSV with the column headers
# output_csv_path = "../../../data/geospatial/stats/watershed_area_stats.csv"
# results_df.to_csv(output_csv_path, index=False)

# print("CSV saved to:", output_csv_path)
results_df

,river_name,total_area,excluded_area,pct_excluded
0,Puelo,9121.633249,53.144432,0.582620
1,Yelcho,11369.277107,2694.269377,23.697807
2,Palena,13147.563085,770.061465,5.857066
3,Cisnes,5116.698637,1276.459601,24.946937
4,Aysen,12181.393611,832.997375,6.838276


In [65]:
# Ensure that the collapsed land cover DataArray has the CRS set.
if not lc_da_collapsed.rio.crs:
    lc_da_collapsed = lc_da_collapsed.rio.write_crs(ws_gdf_proj.crs)

# Define a simplified land cover key for five classes.
# (Labels are cleaned up for the table display.)
landcover_key = {
    1: "       Forest",
    2: "   Cold Steppe",
    3: "  Agriculture",
    4: "Bare/Snow/Ice",
    5: "        Water"
}

# List to store percent cover results for each watershed.
landcover_results = []

# Loop over each watershed in the prepared GeoDataFrame.
for idx, ws in ws_gdf_proj.iterrows():
    ws_name = ws["geometry_name"]
    print(f"Processing watershed: {ws_name}")

    # Convert the watershed geometry to a GeoJSON-like mapping.
    geom = [ws.geometry.__geo_interface__]

    # Clip the collapsed land cover raster to the current watershed.
    try:
        lc_ws = lc_da_collapsed.rio.clip(geom, ws_gdf_proj.crs, all_touched=True, drop=True)
    except Exception as e:
        print(f"Error clipping {ws_name}: {e}")
        continue

    # Flatten the clipped raster and remove no-data (NaN) values.
    pixel_values = lc_ws.values.flatten()
    pixel_values = pixel_values[~np.isnan(pixel_values)]

    if pixel_values.size == 0:
        print(f"No valid pixels found in watershed {ws_name}")
        continue

    total_pixels = pixel_values.size

    # Count occurrences of each land cover class.
    unique_classes, counts = np.unique(pixel_values, return_counts=True)
    class_counts = dict(zip(unique_classes.astype(int), counts))

    # Build a dictionary to store percent cover for the current watershed.
    ws_result = {"Watershed": ws_name}
    for class_val, class_label in landcover_key.items():
        count = class_counts.get(class_val, 0)
        percent_cover = (count / total_pixels) * 100
        ws_result[class_label] = percent_cover

    landcover_results.append(ws_result)

# Create a DataFrame from the collected results and round to 2 decimals.
landcover_percent_df = pd.DataFrame(landcover_results)
landcover_percent_df = landcover_percent_df.round(2)

# Display the results table.
print("Land cover percent by watershed:")
print(landcover_percent_df)

# # Optionally, save the table as a CSV.
# output_csv_path = "../../../data/geospatial/stats/landcover_stats.xlsx"
# landcover_percent_df.to_csv(output_csv_path, index=False)
# print("CSV saved to:", output_csv_path)

Processing watershed: Puelo
Processing watershed: Yelcho
Processing watershed: Palena
Processing watershed: Cisnes
Processing watershed: Aysen
Land cover percent by watershed:
  Watershed         Forest     Cold Steppe    Agriculture  Bare/Snow/Ice  \
0     Puelo          64.85           14.22           0.95          17.18   
1    Yelcho          49.90           26.64           1.97          17.20   
2    Palena          53.20           27.24           0.56          16.31   
3    Cisnes          51.91           29.99           0.04          17.03   
4     Aysen          50.46           26.92           6.13          14.90   

           Water  
0           2.80  
1           4.29  
2           2.69  
3           1.03  
4           1.59  


In [66]:
######### CALCULATE GEOLOGY COVER STATS #####

# Ensure that the geology GeoDataFrame is in the same CRS as the watersheds.
if geo_gdf.crs != ws_gdf_proj.crs:
    geo_gdf = geo_gdf.to_crs(ws_gdf_proj.crs)

# Define the geology categories (based on your standardized geology_group_english).
# Adjust the order if needed.
geology_categories = ["Intrusive", "Metamorphic", "Volcanic-Sedimentary", "Volcanic", "Sedimentary", "Other"]

# List to store the results for each watershed.
geology_results = []

# Loop over each watershed.
for idx, ws in ws_gdf_proj.iterrows():
    ws_name = ws["geometry_name"]
    print(f"Processing watershed: {ws_name}")

    watershed_geom = ws.geometry
    watershed_area = watershed_geom.area  # area in projection units (e.g., m^2)

    # Clip the geology polygons to the current watershed geometry.
    try:
        # geopandas.clip returns only the portions of geo_gdf that intersect with the watershed.
        geo_ws = gpd.clip(geo_gdf, watershed_geom)
    except Exception as e:
        print(f"Error clipping geology for watershed {ws_name}: {e}")
        continue

    # If no geology polygons are found within the watershed, note it and skip.
    if geo_ws.empty:
        print(f"No geology polygons found for watershed {ws_name}")
        continue

    # Create a copy and compute the area of each (clipped) geology polygon.
    geo_ws = geo_ws.copy()
    geo_ws["clipped_area"] = geo_ws.geometry.area

    # Group by the standardized geology category.
    grouped = geo_ws.groupby("geology_group_english")["clipped_area"].sum()

    # Prepare a dictionary for this watershed.
    ws_result = {"Watershed": ws_name}
    for category in geology_categories:
        # Get summed area for this category; if not present, 0.
        area_cat = grouped.get(category, 0)
        percent_cover = (area_cat / watershed_area) * 100
        ws_result[category] = percent_cover
    geology_results.append(ws_result)

# Create a DataFrame from the collected results and round to 2 decimals.
geology_percent_df = pd.DataFrame(geology_results)
geology_percent_df = geology_percent_df.round(2)

print("Geology percent by watershed:")
display(geology_percent_df)

# # Optionally, save the results as an Excel file.
# output_excel_path = "../../../data/geospatial/stats/geology_stats.xlsx"
# geology_percent_df.to_excel(output_excel_path, index=False)
# print("Excel saved to:", output_excel_path)

Processing watershed: Puelo
Processing watershed: Yelcho
Processing watershed: Palena
Processing watershed: Cisnes
Processing watershed: Aysen
Geology percent by watershed:


,Watershed,Intrusive,Metamorphic,Volcanic-Sedimentary,Volcanic,Sedimentary,Other
0,Puelo,38.29,2.9,18.97,3.24,2.25,33.74
1,Yelcho,21.35,0.0,27.93,5.84,8.13,35.77
2,Palena,9.78,0.0,11.79,6.02,15.59,56.35
3,Cisnes,0.02,0.0,0.03,0.03,0.11,99.55
4,Aysen,0.00,0.0,2.28,0.05,2.28,93.98


In [67]:
# ######## COMPUTE AREA EXCLUDED BY YELCHO DAM ##################

# dam_excluded_area = futa_dam_exclude.area/1E6

# yelcho_pct_excluded = (dam_excluded_area / total_area[1]) * 100

# print(f'Percent of yelcho Watershed Captured by Futa Dam: {yelcho_pct_excluded[0]}%')

## Plot Study Area

### GEOGRAPHICAL CONTEXT - WHERE IN THE WORLD ARE WE

### STUDY AREA MAPS, ONE PANEL (PRESENTATION FIGURES)

### EXPLANATORY MAPS, MULTIPANEL (PAPER)

In [68]:
########### SETUP CATOPY CRS FOR PLOTTING WITH LAT/LON TICKS ######################

crs_cartopy = ccrs.AlbersEqualArea(central_longitude=-73.8061523,central_latitude=-44.0918663,standard_parallels=(-46.0556595,-42.128073))
#crs_cartopy

In [69]:
####### CRITICAL SETEUP FOR ALL FOLOWING PLOTTING CELLS WITH GEOLOGY - DO NOT DELETE

# 1) List your geology categories in the same order you coded them:
geology_order = [
    "   Intrusive",
    "Metamorphic",
    " Volc + Sed",
    "     Volcanic",
    "Sedimentary",
    "        Other"
]

# 2) Grab the Set3 palette and pull out 6 distinct colors
set3      = plt.get_cmap("Set3")
geol_colors = [ set3(i) for i in range(len(geology_order)) ]

# 3) Build a ListedColormap from those hex/RGBAs
cmap_geol = mcolors.ListedColormap(geol_colors)

# 4) Create the norm: boundaries at -0.5, 0.5, 1.5, … up to 5.5
norm_geol = mcolors.BoundaryNorm(
    boundaries=np.arange(-0.5, len(geology_order) + 0.5, 1),
    ncolors=len(geology_order)
)

In [70]:
########### PLOTTING CELL VERSION 8: FOUR PANEL SIMPLIFIED FOR PAPER ###########

# --- Assumes ws_gdf_proj already exists in this session, in crs_cartopy ---
# crs_cartopy = ws_gdf_proj.crs.to_string()

# === 1) Clip DEM to watershed bounds (in PlateCarree) ===
dem = rxr.open_rasterio("../data/geospatial/output_SRTM15Plus.tif", masked=True).squeeze()

# Geographic bounds of your watersheds
lon_min_ws, lat_min_ws, lon_max_ws, lat_max_ws = ws_gdf_proj.to_crs("EPSG:4326").total_bounds

dem_clipped = dem.rio.clip_box(
    minx=lon_min_ws, maxx=lon_max_ws,
    miny=lat_min_ws, maxy=lat_max_ws
)


# === 2) Compute hillshade & bathymetry exactly as in Figure 1 ===
elev = dem_clipped.values
lons = dem_clipped['x'].values
lats = dem_clipped['y'].values
dx, dy = map(abs, dem_clipped.rio.resolution())

ls = LightSource(azdeg=315, altdeg=45)
hillshade_all = ls.shade(
    elev, vert_exag=0.6, dx=dx, dy=dy,
    cmap=cm.Greys_r, blend_mode='soft'
)
land_mask = elev >= 0
hillshade_land = np.zeros_like(hillshade_all)
bright_land = np.interp(hillshade_all[land_mask], (0,1), (0.8,1.0))
hillshade_land[land_mask] = bright_land

bathymetry = np.where(elev<0, np.abs(elev), np.nan)
bathymetry = np.clip(bathymetry, 0, 350)
norm_bathy = bathymetry/350
ocean_rgb = cm.Blues(norm_bathy)
ocean_rgb[..., -1] = 0.4

# === 3) Reproject background to your custom CRS ===
xx, yy = np.meshgrid(lons, lats)
transformer = Transformer.from_crs("EPSG:4326", crs_cartopy, always_xy=True)
x_proj, y_proj = transformer.transform(xx, yy)
extent_proj = [x_proj.min(), x_proj.max(), y_proj.min(), y_proj.max()]

# === 4) Plot panels ===
fig, axs = plt.subplots(
    1, 4, figsize=(12,10), sharex=True, sharey=True,
    subplot_kw={'projection': crs_cartopy}, dpi=300
)
plt.subplots_adjust(left=0.15, wspace=0.0)

# Draw bathy + hillshade backdrop
for ax in axs:
    ax.set_extent(extent_proj, crs=crs_cartopy)
    ax.imshow(ocean_rgb, origin='upper',
              extent=extent_proj, transform=crs_cartopy, zorder=0)
    ax.imshow(hillshade_land, origin='upper',
              extent=extent_proj, transform=crs_cartopy, zorder=1)

# Panel 1: Elevation
dem_da_clipped.plot.imshow(
    ax=axs[0], zorder=3, vmin=0, vmax=2500, cmap='inferno', alpha=1,
    cbar_kwargs={"orientation":"horizontal","label":"meters above sea level",
                "shrink":0.8,"fraction":0.07,"pad":0.09}
)
ws_gdf_proj.plot(ax=axs[0], zorder=4, facecolor='none', edgecolor='k', linewidth=1.2)
axs[0].set_title("Elevation", fontsize=14)

# Panel 2: Land Cover
im = lc_da_collapsed.plot.imshow(
    ax=axs[1], zorder=3, cmap=cmap_collapsed, norm=norm, alpha=0.9,
    cbar_kwargs={"orientation":"horizontal","label":"","shrink":0.8,
                "fraction":0.07,"pad":0.09}
)
ws_gdf_proj.plot(ax=axs[1], zorder=4, facecolor='none', edgecolor='k', linewidth=1.2)
cbar = im.colorbar; cbar.ax.set_xticks([])
x_off, y_off = -0.1, -1.4
for m in range(1,6):
    lbl = landcover_key[m]
    rx = (m-0.55)/5.0 -0.1 + x_off
    ry = -3.2 + y_off
    cbar.ax.text(rx, ry, lbl, rotation=45, ha='center', va='center',
                 fontsize=10, transform=cbar.ax.transAxes)
axs[1].set_title("Land Cover", fontsize=14)

# Panel 3: Precipitation
prec_clip_annual.plot.imshow(
    ax=axs[2], zorder=3, robust=True, cmap="Blues", alpha=1,
    transform=crs_cartopy,
    cbar_kwargs={"orientation":"horizontal","label":"millimeters per year",
                "shrink":0.8,"fraction":0.07,"pad":0.09}
)
ws_gdf_proj.plot(ax=axs[2], zorder=4, facecolor='none', edgecolor='black', linewidth=0)
ws_gdf_proj.boundary.plot(ax=axs[2], zorder=4, color='black', linewidth=1.2)
axs[2].set_title("Precipitation", fontsize=14)

# Panel 4: Geology
slope_da.plot.imshow(
    ax=axs[3], zorder=1, cmap="viridis", robust=True,
    alpha=0, transform=crs_cartopy, add_colorbar=False
)
ws_gdf_proj.boundary.plot(ax=axs[3], zorder=4, color='black', linewidth=1.2)
geo_gdf.plot(ax=axs[3], zorder=3,
             column='geology_code', cmap=cmap_geol, norm=norm_geol,
             legend=False, edgecolor='none', alpha=0.8)
sm = ScalarMappable(cmap=cmap_geol, norm=norm_geol); sm.set_array([])
cbar_geo = fig.colorbar(sm, ax=axs[3], orientation="horizontal",
                        shrink=0.8, fraction=0.07, pad=0.09)
cbar_geo.ax.set_xticks([])
xo, yo = -0.08, -1.45
geol_labels = ["   Intrusive","Metamorphic"," Volc + Sed",
               "     Volcanic","Sedimentary","        Other"]
for m, lbl in enumerate(geol_labels, start=1):
    rx = (m-0.55)/6.0 -0.1 + xo
    ry = -3.2 + yo
    cbar_geo.ax.text(rx, ry, lbl, rotation=45,
                     ha='center', va='center',
                     fontsize=10, transform=cbar_geo.ax.transAxes)
axs[3].set_title("Geology", fontsize=14)

# Clean up axes
for ax in axs:
    ax.set_xticks([]); ax.set_yticks([]); ax.set_xlabel(""); ax.set_ylabel("")

if savefig:
    plt.savefig(fig_path/"study_area_4p_simple.png", dpi=300, bbox_inches="tight")
plt.show()


Output hidden; open in https://colab.research.google.com to view.

### END OF EXPLORATORY MAPS - ALL BELOW IS OLD CODE

## OLD CODE: "Now we want to calculate some stats for the landcover data" (KEEP THIS TOO)

## OLD CODE: "Now we analyze the DEM data" (KEEP THIS TOO)